# Pakistani Law RAG — Data Collection, Chunking & Pinecone Upsert

**Run this notebook once.** It will:
1. Scrape Pakistani legal texts from public sources
2. Clean and chunk them two ways (Fixed + Recursive) for the ablation study
3. Embed all chunks with `all-MiniLM-L6-v2`
4. Upsert both chunk sets into Pinecone under separate namespaces
5. Save chunk JSONs locally (needed for BM25 in the app)

**Before running:** Make sure you have selected `Runtime > Change runtime type > T4 GPU`

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q pinecone sentence-transformers langchain langchain-community \
                 beautifulsoup4 requests pdfplumber rank-bm25 tqdm tiktoken \
                 lxml html5lib

---
## Cell 2 — Configuration (EDIT THIS CELL)

In [ ]:
# ============================================================
#  FILL IN YOUR OWN VALUES HERE
# ============================================================

PINECONE_API_KEY  = "YOUR_PINECONE_API_KEY_HERE"   # from pinecone.io dashboard
PINECONE_INDEX    = "pakistani-law"                 # the index name you created

# Namespaces — one per chunking strategy (for ablation study)
NS_FIXED     = "fixed"
NS_RECURSIVE = "recursive"

# Chunking parameters
FIXED_CHUNK_SIZE    = 256    # tokens
FIXED_CHUNK_OVERLAP = 30     # tokens
REC_CHUNK_SIZE      = 256    # characters (RecursiveCharacterTextSplitter uses chars)
REC_CHUNK_OVERLAP   = 50

# Embedding model — produces 384-dim vectors, matches your Pinecone index
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBED_BATCH     = 64

print("Config loaded.")

---
## Cell 3 — Import Libraries

In [ ]:
import requests
import time
import json
import re
import os
from bs4 import BeautifulSoup
from tqdm import tqdm
import tiktoken
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pinecone import Pinecone

print("All imports successful.")

---
## Cell 4 — Define the Corpus (Pakistani Law Acts)

Each entry is a URL from `pakistancode.gov.pk` — the official public registry of Pakistani legislation.
We scrape the HTML directly. No login required, all public domain.

In [ ]:
CORPUS = [
    # (display_name, year, url)
    # These URLs hit the Pakistan Code official site HTML viewer
    ("Constitution of Pakistan 1973",           1973, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJl="),
    ("Pakistan Penal Code 1860",                1860, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJm="),
    ("Code of Criminal Procedure 1898",         1898, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJn="),
    ("Civil Procedure Code 1908",               1908, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJo="),
    ("Contract Act 1872",                       1872, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJp="),
    ("Qanun-e-Shahadat Order 1984",             1984, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJq="),
    ("Anti-Terrorism Act 1997",                 1997, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJr="),
    ("National Accountability Ordinance 1999",  1999, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJs="),
    ("Muslim Family Laws Ordinance 1961",       1961, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJt="),
    ("Companies Act 2017",                      2017, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJu="),
    ("Income Tax Ordinance 2001",               2001, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJv="),
    ("Arbitration Act 1940",                    1940, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJw="),
]

# ----------------------------------------------------------------
# FALLBACK DIRECT TEXT SOURCES (more reliable than scraping)
# If the scraper fails for an act, we also define fallback plain-
# text URLs from commonlii.org which has clean HTML for Pakistani law
# ----------------------------------------------------------------
FALLBACK_CORPUS = [
    ("Pakistan Penal Code 1860",           1860, "https://www.commonlii.org/pk/legis/num_act/ppc1860132/"),
    ("Constitution of Pakistan 1973",      1973, "https://www.commonlii.org/pk/legis/const/2012/"),
    ("Contract Act 1872",                  1872, "https://www.commonlii.org/pk/legis/num_act/ca1872149/"),
    ("Code of Criminal Procedure 1898",    1898, "https://www.commonlii.org/pk/legis/num_act/cocp1898230/"),
    ("Qanun-e-Shahadat Order 1984",        1984, "https://www.commonlii.org/pk/legis/num_act/qeo1984268/"),
]

print(f"Corpus defined: {len(CORPUS)} primary + {len(FALLBACK_CORPUS)} fallback sources.")

---
## Cell 5 — Scraper Functions

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

def clean_text(raw: str) -> str:
    """Remove excess whitespace, page markers, and non-printable chars."""
    # Collapse multiple newlines to double
    text = re.sub(r'\n{3,}', '\n\n', raw)
    # Remove page number patterns like 'Page 1 of 45'
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)
    # Remove lines that are just numbers (page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    # Remove excessive spaces
    text = re.sub(r' {3,}', ' ', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text


def scrape_commonlii(url: str, name: str, year: int) -> dict | None:
    """
    Scrape a legal document from commonlii.org.
    Returns a document dict or None on failure.
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        # commonlii puts the main content inside <div class="judgment"> or <body>
        # Try judgment div first, fall back to body
        content = soup.find('div', class_='judgment')
        if not content:
            content = soup.find('body')
        if not content:
            return None

        # Remove navigation, scripts, styles
        for tag in content.find_all(['script', 'style', 'nav', 'header', 'footer', 'noscript']):
            tag.decompose()

        raw_text = content.get_text(separator='\n')
        text = clean_text(raw_text)

        if len(text) < 1000:   # Too short — probably a failed scrape
            return None

        return {
            "name":   name,
            "year":   year,
            "url":    url,
            "text":   text,
            "length": len(text)
        }

    except Exception as e:
        print(f"  FAILED ({name}): {e}")
        return None


def scrape_pakistancode(url: str, name: str, year: int) -> dict | None:
    """
    Scrape from pakistancode.gov.pk.
    The site renders content inside <div id='content'> or similar.
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        # Try common content containers
        content = (soup.find('div', id='content') or
                   soup.find('div', class_='content') or
                   soup.find('div', class_='act-content') or
                   soup.find('main') or
                   soup.find('article'))

        if not content:
            content = soup.find('body')

        for tag in content.find_all(['script', 'style', 'nav', 'header', 'footer']):
            tag.decompose()

        raw_text = content.get_text(separator='\n')
        text = clean_text(raw_text)

        if len(text) < 1000:
            return None

        return {
            "name":   name,
            "year":   year,
            "url":    url,
            "text":   text,
            "length": len(text)
        }

    except Exception as e:
        print(f"  FAILED ({name}): {e}")
        return None


print("Scraper functions defined.")

---
## Cell 6 — Run the Scraper

This first tries `commonlii.org` (cleaner HTML), then falls back to `pakistancode.gov.pk`.
Be patient — it sleeps 1-2s between requests to avoid being blocked.

In [ ]:
documents = []
failed    = []

print("Scraping fallback corpus (commonlii.org — cleaner HTML)...")
for name, year, url in tqdm(FALLBACK_CORPUS, desc="Scraping"):
    doc = scrape_commonlii(url, name, year)
    if doc:
        documents.append(doc)
        print(f"  OK  {name} — {doc['length']:,} chars")
    else:
        failed.append((name, year, url))
        print(f"  FAIL {name}")
    time.sleep(1.5)

# For documents that failed, try pakistancode.gov.pk
if failed:
    print(f"\nRetrying {len(failed)} failed docs on pakistancode.gov.pk...")
    for name, year, url in failed:
        # Find matching URL in primary corpus
        match = next((u for n, y, u in CORPUS if n == name), None)
        if match:
            doc = scrape_pakistancode(match, name, year)
            if doc:
                documents.append(doc)
                print(f"  OK  {name} — {doc['length']:,} chars")
        time.sleep(2)

print(f"\n=== Scraped {len(documents)} documents ===")
total_chars = sum(d['length'] for d in documents)
print(f"Total text: {total_chars:,} characters (~{total_chars//5:,} words)")

# Save raw docs as backup
with open("raw_documents.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)
print("Raw documents saved to raw_documents.json")

---
## Cell 7 — Manual Text Entry (Run if scraping failed badly)

If you got fewer than 3 documents from the scraper, run this cell.
It loads from text files you upload manually.

**Instructions:**
1. Go to `pakistancode.gov.pk`, open any act, Ctrl+A, Ctrl+C, paste into a .txt file
2. Upload the .txt files to this Colab session using the Files panel on the left
3. Run this cell — it will load whatever .txt files are present

In [ ]:
import glob

txt_files = glob.glob("*.txt")
if txt_files:
    for path in txt_files:
        name = os.path.splitext(os.path.basename(path))[0].replace('_', ' ')
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            text = clean_text(f.read())
        if len(text) > 500:
            doc = {"name": name, "year": 0, "url": path, "text": text, "length": len(text)}
            # Avoid duplicates
            existing_names = [d['name'] for d in documents]
            if name not in existing_names:
                documents.append(doc)
                print(f"Loaded from file: {name} — {len(text):,} chars")
else:
    print("No .txt files found. Skipping manual entry.")

print(f"\nTotal documents available: {len(documents)}")

---
## Cell 8 — Chunking Strategy A: Fixed Size (Token-Based)

Uses `tiktoken` to count tokens accurately. Slides a window of 256 tokens with 30-token overlap.
This is the baseline — simple, predictable, ignores sentence boundaries.

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")  # Same tokenizer used by many models

def fixed_chunk(text: str, chunk_size: int = 256, overlap: int = 30) -> list[str]:
    """
    Split text into chunks of exactly `chunk_size` tokens with `overlap` token overlap.
    Returns list of text strings.
    """
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = enc.decode(chunk_tokens)
        if len(chunk_text.strip()) > 50:  # skip tiny fragments
            chunks.append(chunk_text.strip())
        start += chunk_size - overlap
    return chunks


fixed_chunks = []
for doc in documents:
    doc_chunks = fixed_chunk(doc['text'], FIXED_CHUNK_SIZE, FIXED_CHUNK_OVERLAP)
    for i, chunk_text in enumerate(doc_chunks):
        fixed_chunks.append({
            "id":       f"fixed-{doc['name'][:30].replace(' ', '-')}-{i:04d}",
            "text":     chunk_text,
            "source":   doc['name'],
            "year":     doc['year'],
            "url":      doc['url'],
            "strategy": "fixed",
            "chunk_idx": i
        })

# Clean IDs — Pinecone IDs cannot have special chars except - and _
for c in fixed_chunks:
    c['id'] = re.sub(r'[^a-zA-Z0-9\-_]', '', c['id'])

print(f"Fixed chunking produced: {len(fixed_chunks)} chunks")
print(f"Sample chunk:\n---\n{fixed_chunks[0]['text'][:300]}\n---")

with open("chunks_fixed.json", "w", encoding="utf-8") as f:
    json.dump(fixed_chunks, f, ensure_ascii=False, indent=2)
print("Saved to chunks_fixed.json")

---
## Cell 9 — Chunking Strategy B: Recursive (Structure-Aware)

Splits on natural boundaries in order: paragraph breaks → newlines → sentences → words.
Only falls to the next level if the current chunk is still too large.
Produces more semantically coherent chunks that respect section boundaries.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    chunk_size=1200,       # ~256 tokens in characters
    chunk_overlap=150,
    length_function=len,
)

recursive_chunks = []
for doc in documents:
    doc_chunks = splitter.split_text(doc['text'])
    for i, chunk_text in enumerate(doc_chunks):
        if len(chunk_text.strip()) < 50:
            continue
        recursive_chunks.append({
            "id":       f"rec-{doc['name'][:30].replace(' ', '-')}-{i:04d}",
            "text":     chunk_text.strip(),
            "source":   doc['name'],
            "year":     doc['year'],
            "url":      doc['url'],
            "strategy": "recursive",
            "chunk_idx": i
        })

for c in recursive_chunks:
    c['id'] = re.sub(r'[^a-zA-Z0-9\-_]', '', c['id'])

print(f"Recursive chunking produced: {len(recursive_chunks)} chunks")
print(f"Sample chunk:\n---\n{recursive_chunks[0]['text'][:300]}\n---")

with open("chunks_recursive.json", "w", encoding="utf-8") as f:
    json.dump(recursive_chunks, f, ensure_ascii=False, indent=2)
print("Saved to chunks_recursive.json")

print(f"\n=== Chunk Summary ===")
print(f"Fixed chunks:     {len(fixed_chunks)}")
print(f"Recursive chunks: {len(recursive_chunks)}")
print(f"Total vectors to upsert: {len(fixed_chunks) + len(recursive_chunks)}")

---
## Cell 10 — Load Embedding Model

Downloads `all-MiniLM-L6-v2` (~80MB). Uses GPU if available.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)

# Quick sanity check
test_vec = embed_model.encode("Test sentence for Pakistani law system.")
print(f"Embedding model loaded. Output dimension: {len(test_vec)} (expected 384)")

---
## Cell 11 — Generate Embeddings for Both Chunk Sets

In [ ]:
def embed_chunks(chunks: list[dict], batch_size: int = 64) -> list[list[float]]:
    """Embed all chunks and return as list of float lists."""
    texts = [c['text'] for c in chunks]
    embeddings = embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    return embeddings.tolist()


print("Embedding fixed chunks...")
fixed_embeddings = embed_chunks(fixed_chunks, EMBED_BATCH)
print(f"Done. Shape: {len(fixed_embeddings)} x {len(fixed_embeddings[0])}")

print("\nEmbedding recursive chunks...")
recursive_embeddings = embed_chunks(recursive_chunks, EMBED_BATCH)
print(f"Done. Shape: {len(recursive_embeddings)} x {len(recursive_embeddings[0])}")

---
## Cell 12 — Connect to Pinecone

In [ ]:
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)

stats = index.describe_index_stats()
print(f"Connected to Pinecone index: '{PINECONE_INDEX}'")
print(f"Current vector count: {stats.total_vector_count}")
print(f"Namespaces present:   {list(stats.namespaces.keys())}")

---
## Cell 13 — Upsert to Pinecone

Upserts in batches of 100 (Pinecone's recommended batch size).
Metadata stored alongside each vector includes the raw chunk text —
this is what gets retrieved and shown to the LLM.

In [ ]:
def upsert_to_pinecone(chunks: list[dict],
                       embeddings: list[list[float]],
                       namespace: str,
                       batch_size: int = 100):
    """
    Upsert chunks + embeddings into a Pinecone namespace.
    Metadata includes text, source, year, url, strategy.
    """
    total   = len(chunks)
    upserted = 0

    for start in tqdm(range(0, total, batch_size), desc=f"Upserting [{namespace}]"):
        end   = min(start + batch_size, total)
        batch = []

        for i in range(start, end):
            chunk = chunks[i]
            # Pinecone metadata values must be str, int, float, or bool
            # Truncate text to 40960 chars (Pinecone metadata limit)
            metadata = {
                "text":      chunk['text'][:40000],
                "source":    chunk['source'],
                "year":      int(chunk['year']),
                "url":       chunk['url'],
                "strategy":  chunk['strategy'],
                "chunk_idx": int(chunk['chunk_idx'])
            }
            batch.append({
                "id":       chunk['id'],
                "values":   embeddings[i],
                "metadata": metadata
            })

        index.upsert(vectors=batch, namespace=namespace)
        upserted += len(batch)

    print(f"Upserted {upserted} vectors into namespace '{namespace}'")


# --- Upsert fixed chunks ---
print("=" * 50)
upsert_to_pinecone(fixed_chunks, fixed_embeddings, NS_FIXED)

# --- Upsert recursive chunks ---
print("=" * 50)
upsert_to_pinecone(recursive_chunks, recursive_embeddings, NS_RECURSIVE)

---
## Cell 14 — Verify the Upsert

In [ ]:
time.sleep(5)  # Wait for Pinecone to index

stats = index.describe_index_stats()
print("=== Pinecone Index Stats ===")
print(f"Total vectors: {stats.total_vector_count}")
for ns, ns_stats in stats.namespaces.items():
    print(f"  Namespace '{ns}': {ns_stats.vector_count} vectors")

---
## Cell 15 — Quick Retrieval Test

Run a sample query to verify the pipeline is working end to end before moving on.

In [ ]:
test_query = "What is the punishment for murder under Pakistani law?"

query_vec = embed_model.encode(test_query).tolist()

results = index.query(
    vector=query_vec,
    top_k=3,
    namespace=NS_FIXED,
    include_metadata=True
)

print(f"Query: {test_query}\n")
print("Top 3 results from Pinecone (fixed namespace):")
print("=" * 60)
for i, match in enumerate(results.matches):
    print(f"\n[{i+1}] Score: {match.score:.4f}")
    print(f"     Source: {match.metadata['source']}")
    print(f"     Text preview: {match.metadata['text'][:250]}...")

---
## Cell 16 — Download Your Chunk Files

Download both JSON files to your local machine. You will need them when building the Gradio app
(BM25 runs from these files — it is not stored in Pinecone).

In [ ]:
from google.colab import files

print("Downloading chunk files...")
files.download("chunks_fixed.json")
files.download("chunks_recursive.json")
files.download("raw_documents.json")
print("Done. Save these files — you will upload them to HF Spaces later.")

---
## Cell 17 — Summary

Run this at the end to confirm everything is in order.

In [ ]:
print("╔══════════════════════════════════════════════════╗")
print("║         DATA & EMBEDDINGS — COMPLETE             ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Documents scraped:       {len(documents):<24}║")
print(f"║  Fixed chunks:            {len(fixed_chunks):<24}║")
print(f"║  Recursive chunks:        {len(recursive_chunks):<24}║")
print(f"║  Pinecone index:          {PINECONE_INDEX:<24}║")
print(f"║  Namespace [fixed]:       {NS_FIXED:<24}║")
print(f"║  Namespace [recursive]:   {NS_RECURSIVE:<24}║")
print("╠══════════════════════════════════════════════════╣")
print("║  Next step: build retrieval pipeline             ║")
print("║  (BM25 + Semantic + RRF + CrossEncoder)          ║")
print("╚══════════════════════════════════════════════════╝")